In [63]:
import matplotlib.pyplot as plt
import math
from math import log
import pandas as pd
import numpy as np
import random

### Load data

#### MovieLens

In [4]:
df_links = pd.read_csv('data/movielens/ml-latest-small/links.csv')
df_movies = pd.read_csv('data/movielens/ml-latest-small/movies.csv')
df_ratings = pd.read_csv('data/movielens/ml-latest-small/ratings.csv')
df_tags = pd.read_csv('data/movielens/ml-latest-small/tags.csv')

In [5]:
df_tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


#### Instacart

In [64]:
#instacart
df_products = pd.read_csv('data/instacart/products.csv')
df_carts_prior = pd.read_csv('data/instacart/order_products__prior.csv')
df_carts_train = pd.read_csv('data/instacart/order_products__train.csv')
df_carts = pd.concat([df_carts_prior, df_carts_train])
df_carts.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [65]:
df_carts.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [82]:
# transforms the data into a long list of lists, each containing product IDs 
# that were purchased in the same order
carts = df_carts[['order_id', 'product_id']].groupby('order_id')['product_id'].apply(list).to_list()
carts[0:3]
print(len(carts))

3346083


In [103]:
small_carts = carts[:1000000]
print(len(small_carts))

1000000


### Ranking

Recommend 10 movies based on recent ratings

Use techniques to make sure the recommendation is reliable

In [10]:
# 
df_last_ten = df_ratings[df_ratings['timestamp'] > 1456233838 ]
df_last_ten.head()

,userId,movieId,rating,timestamp
1434,15,1,2.5,1510577970
1436,15,47,3.5,1510571970
1440,15,260,5.0,1510571946
1441,15,293,3.0,1510571962
1442,15,296,4.0,1510571877


In [11]:
avg_rating_past_ten = df_last_ten['rating'].mean()
print(avg_rating_past_ten)

3.4770202143801088


In [26]:
summed_rating = df_last_ten.groupby(['movieId'])['rating'].sum()
count_rating = df_last_ten.groupby(['movieId'])['rating'].count()
#summed_rating.head()
#count_rating.head()

df_sumrating_and_countrating = df_last_ten.groupby(['movieId']).agg(
  sum_ratings = ('rating', 'sum'),
  count_ratings = ('rating', 'count')
)

print(df_sumrating_and_countrating)


         sum_ratings  count_ratings
movieId                            
1              121.0             31
2               54.5             15
3                1.5              1
5                6.5              2
6               52.5             12
...              ...            ...
193581           4.0              1
193583           3.5              1
193585           3.5              1
193587           3.5              1
193609           4.0              1

[5673 rows x 2 columns]


In [50]:
k = 10
test = df_sumrating_and_countrating.copy()
test['Damped Mean'] = test.apply(lambda row: (row['sum_ratings'] + k*avg_rating_past_ten) / (row['count_ratings'] + k), axis = 1)

print(test)

         sum_ratings  count_ratings  Damped Mean
movieId                                         
1              121.0             31     3.799273
2               54.5             15     3.570808
3                1.5              1     3.297291
5                6.5              2     3.439184
6               52.5             12     3.966827
...              ...            ...          ...
193581           4.0              1     3.524564
193583           3.5              1     3.479109
193585           3.5              1     3.479109
193587           3.5              1     3.479109
193609           4.0              1     3.524564

[5673 rows x 3 columns]


In [61]:
df_top_10 = test.sort_values(by='Damped Mean', ascending=False).head(10)
#display(df_top_10)

final = pd.merge(df_top_10, df_movies, on='movieId', how='left')
final2 = pd.merge(df_top_10, df_movies[['movieId', 'title']], on='movieId', how='left')

#print(final.head(10))
# Select only the columns you want to display
result_list = final2[['title', 'Damped Mean']]
display(result_list)
display(final)


,title,Damped Mean
0,"Shawshank Redemption, The (1994)",4.311921
1,Fight Club (1999),4.163019
2,"Shining, The (1980)",4.149069
3,Good Will Hunting (1997),4.127963
4,"Godfather: Part II, The (1974)",4.109007
5,Pulp Fiction (1994),4.106856
6,Léon: The Professional (a.k.a. The Professiona...,4.096182
7,WALL·E (2008),4.088337
8,There Will Be Blood (2007),4.063510
9,"Departed, The (2006)",4.063061


,movieId,sum_ratings,count_ratings,Damped Mean,title,genres
0,318,280.0,63,4.311921,"Shawshank Redemption, The (1994)",Crime|Drama
1,2959,227.5,53,4.163019,Fight Club (1999),Action|Crime|Drama|Thriller
2,1258,98.0,22,4.149069,"Shining, The (1980)",Horror
3,1704,167.5,39,4.127963,Good Will Hunting (1997),Drama|Romance
4,1221,88.5,20,4.109007,"Godfather: Part II, The (1974)",Crime|Drama
5,296,187.0,44,4.106856,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
6,293,104.5,24,4.096182,Léon: The Professional (a.k.a. The Professiona...,Action|Crime|Drama|Thriller
7,60069,186.0,44,4.088337,WALL·E (2008),Adventure|Animation|Children|Romance|Sci-Fi
8,56782,46.5,10,4.063510,There Will Be Blood (2007),Drama|Western
9,48516,111.5,26,4.063061,"Departed, The (2006)",Crime|Drama|Thriller


## Reflections
The variable we modify is k, which is the prior strength. If k is set to 0 we just calculate the mean of the rating based on the total amount of ratings. The higher we set k, the more conservative our approach is (it will take us longer to converge to the true mean). 

In this setting we should consider how many users we have, and then how many ratings a movie "should" have. Essentially, if we have 10 million users, a movie can have a super high avg rating (5.0), but if it only has 5 reviews, we would probably want to rank it lower than a movie with a 1000 reviews and an avg rating of (4.5). A lower k in this setting "favors" the avg rating of reviews even if they are few, and a higher k "favors" having more reviews on a movie even if the avg rating is slightly lower. Setting k extremly high would result in all movies basically having a damped_mean converging towards the global_avg_rating, because a movie would need a ton of reviews that are substantially different from the global_avg_rating for its damped_mean to be different. 

### Association rule mining

Calculate the number of frequent itemsets with varying levels for support

Try to guess what value of minimum support would be reasonable

Calculate association rules and find the one whose subsequent item has the least support (the one more in the tail)

#### Priori (Apyori)

In [69]:
from apyori import apriori

## Reflection
Instacart
• Calculate the number of frequent itemsets with varying levels for
support
• Try to guess what value of minimum support would be reasonable
• Calculate association rules and find the one whose subsequent
item has the least support (the one more in the tail)

In [104]:
association_rules = apriori(small_carts, min_support=0.0035, 
                            min_confidence=0.2,
                            min_lift=3, min_length=2)
association_rules = list(association_rules)

In [109]:
print(association_rules)
print(len(association_rules))

[RelationRecord(items=frozenset({47209, 5876}), support=0.006611, ordered_statistics=[OrderedStatistic(items_base=frozenset({5876}), items_add=frozenset({47209}), confidence=0.24169195335063795, lift=3.659725827147347)]), RelationRecord(items=frozenset({43352, 16797}), support=0.003865, ordered_statistics=[OrderedStatistic(items_base=frozenset({43352}), items_add=frozenset({16797}), confidence=0.2104661293835766, lift=4.717913682662555)]), RelationRecord(items=frozenset({26209, 20114}), support=0.003555, ordered_statistics=[OrderedStatistic(items_base=frozenset({20114}), items_add=frozenset({26209}), confidence=0.2682209144409235, lift=6.137778362492528)]), RelationRecord(items=frozenset({21137, 27966}), support=0.010548, ordered_statistics=[OrderedStatistic(items_base=frozenset({27966}), items_add=frozenset({21137}), confidence=0.24960363472869687, lift=3.020921449061384)]), RelationRecord(items=frozenset({39928, 21137}), support=0.003853, ordered_statistics=[OrderedStatistic(items_ba

In [108]:
idx = 5 #prints the 5th association rule

rule = association_rules[idx]
frequent_itemset = rule.items
support = rule.support

antecedent = rule.ordered_statistics[0].items_base
antecedent = [df_products.iloc[a-1]['product_name'] for a in antecedent]
consequent = rule.ordered_statistics[0].items_add
consequent = [df_products.iloc[c-1]['product_name'] for c in consequent]
lift = rule.ordered_statistics[0].lift
confidence = rule.ordered_statistics[0].confidence

print(f'{antecedent}->{consequent}')
print(f'support = {support}')
print(f'confidence = {confidence}')
print(f'lift = {lift}')

['Bunched Cilantro']->['Limes']
support = 0.003801
confidence = 0.2695553506843486
lift = 6.1683146609690755


#### FP-growth (mlxtend)

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpmax, fpgrowth
from mlxtend.frequent_patterns import association_rules

In [ ]:
# encode the dataset into a orders x items binary sparse matrix
te = TransactionEncoder()
te_data = te.fit(carts).transform(carts, sparse=True)
df = pd.DataFrame.sparse.from_spmatrix(te_data, columns=te.columns_)
# product indices must either start from 0 or be strings
df.columns = [str(i) for i in df.columns] 
# alternatively, reduce ids by 1
#carts_modified = [[carts[l][i]-1 for i in range(0, len(carts[l]))] for l in range(0, len(carts))]

In [ ]:
frequent_itemsets = fpgrowth(df, min_support=xxxx, use_colnames=True, verbose=1)

In [ ]:
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=xxxx)
rules